# Week 4 — Data Cleaning and Visualization

**Dataset:** Kaggle *Cervical Cancer Risk Classification* (`kag_risk_factors_cervical_cancer.csv`), 858 rows × 36 columns. A practice dataset: the goal is cleaning and visualization skills, not medical conclusions.

**Plan of the week**
- Monday — missing values
- Tuesday — duplicates, wrong types, row-level problems
- Wednesday — outliers
- Thursday — feature scaling
- Friday — Plotly charts
- Saturday — mini-project: raw data → clean data + 4 charts

## Setup and data loading

In [134]:
import pandas as pd
import numpy as np

In [109]:
# Changing '?' symbols to NaN values, for better working with data
df = pd.read_csv("kag_risk_factors_cervical_cancer.csv", na_values=['?'])

---
## Monday - Missing values

### 1. First look at the data

In [110]:
print(df.shape)
df.head()

(858, 36)


,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),IUD,...,STDs: Time since first diagnosis,STDs: Time since last diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology,Biopsy
0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0,0,0,0,0,0,0,0
1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0,0,0,0,0,0,0,0
2,34,1.0,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0,0,0,0,0,0,0,0
3,52,5.0,16.0,4.0,1.0,37.0,37.0,1.0,3.0,0.0,...,NaN,NaN,1,0,1,0,0,0,0,0
4,46,3.0,21.0,4.0,0.0,0.0,0.0,1.0,15.0,0.0,...,NaN,NaN,0,0,0,0,0,0,0,0


In [140]:
df.median()

Age                                   25.0
Number of sexual partners              2.0
First sexual intercourse              17.0
Num of pregnancies                     2.0
Smokes                                 0.0
Smokes (years)                         0.0
Smokes (packs/year)                    0.0
Hormonal Contraceptives                1.0
Hormonal Contraceptives (years)        0.5
IUD                                    0.0
IUD (years)                            0.0
STDs                                   0.0
STDs (number)                          0.0
STDs:condylomatosis                    0.0
STDs:cervical condylomatosis           0.0
STDs:vaginal condylomatosis            0.0
STDs:vulvo-perineal condylomatosis     0.0
STDs:syphilis                          0.0
STDs:pelvic inflammatory disease       0.0
STDs:genital herpes                    0.0
STDs:molluscum contagiosum             0.0
STDs:AIDS                              0.0
STDs:HIV                               0.0
STDs:Hepati

In [139]:
total_cells = np.prod(df.shape)
total_missing = df.isnull().sum().sum()

percent_missing = (total_missing / total_cells) * 100
print(f'The percentage of total missing data in dataframe: {percent_missing.round()}%')

The percentage of total missing data in dataframe: 12.0%


In [111]:
df.columns

Index(['Age', 'Number of sexual partners', 'First sexual intercourse',
       'Num of pregnancies', 'Smokes', 'Smokes (years)', 'Smokes (packs/year)',
       'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'IUD',
       'IUD (years)', 'STDs', 'STDs (number)', 'STDs:condylomatosis',
       'STDs:cervical condylomatosis', 'STDs:vaginal condylomatosis',
       'STDs:vulvo-perineal condylomatosis', 'STDs:syphilis',
       'STDs:pelvic inflammatory disease', 'STDs:genital herpes',
       'STDs:molluscum contagiosum', 'STDs:AIDS', 'STDs:HIV',
       'STDs:Hepatitis B', 'STDs:HPV', 'STDs: Number of diagnosis',
       'STDs: Time since first diagnosis', 'STDs: Time since last diagnosis',
       'Dx:Cancer', 'Dx:CIN', 'Dx:HPV', 'Dx', 'Hinselmann', 'Schiller',
       'Citology', 'Biopsy'],
      dtype='str')

### 2. Overview of the gaps
Count and percentage of missing values per column.

In [112]:
df.isnull().sum()

Age                                     0
Number of sexual partners              26
First sexual intercourse                7
Num of pregnancies                     56
Smokes                                 13
Smokes (years)                         13
Smokes (packs/year)                    13
Hormonal Contraceptives               108
Hormonal Contraceptives (years)       108
IUD                                   117
IUD (years)                           117
STDs                                  105
STDs (number)                         105
STDs:condylomatosis                   105
STDs:cervical condylomatosis          105
STDs:vaginal condylomatosis           105
STDs:vulvo-perineal condylomatosis    105
STDs:syphilis                         105
STDs:pelvic inflammatory disease      105
STDs:genital herpes                   105
STDs:molluscum contagiosum            105
STDs:AIDS                             105
STDs:HIV                              105
STDs:Hepatitis B                  

In [113]:
missing_values = df.isnull().mean() * 100
missing_values.sort_values(ascending=False)
""" The worst column is 'STDs: Time since first diagnosis' and 'STDs: Time since last diagnosis' with almost 92% of leak.
This information wasn't recorded, so we should save it and find the way how to return any value.
As a measure, we may try to guess the approximate time from the first and last diagnosis of a patient."""

" The worst column is 'STDs: Time since first diagnosis' and 'STDs: Time since last diagnosis' with almost 92% of leak.\nThis information wasn't recorded, so we should save it and find the way how to return any value.\nAs a measure, we may try to guess the approximate time from the first and last diagnosis of a patient."

### 3. Investigating the worst columns
`STDs: Time since first diagnosis` and `STDs: Time since last diagnosis` (~92% missing).
Question: are these gaps *not recorded* or *non-existent* (no diagnosis → no time)?

#### 3.1 Does `STDs: Number of diagnosis` have gaps?

In [114]:
df['STDs: Number of diagnosis'].isnull()

0      False
1      False
2      False
3      False
4      False
       ...  
853    False
854    False
855    False
856    False
857    False
Name: STDs: Number of diagnosis, Length: 858, dtype: bool

#### 3.2 Gap in `Time since first diagnosis` vs `Number of diagnosis` (crosstab)

In [115]:
pd.crosstab(df['STDs: Time since first diagnosis'].isnull(), df['STDs: Number of diagnosis'], rownames=['missing time'], colnames=['time exists'])

time exists,0,1,2,3
missing time,,,,
False,0,68,2,1
True,787,0,0,0


#### 3.3 Are the gaps in the two time columns in the same rows?

In [116]:
(df['STDs: Time since first diagnosis'].notnull() == df['STDs: Time since last diagnosis'].notnull()).sum()

np.int64(858)

#### 3.4 Are the existing values in the two time columns equal?
Then: which rows differ?

In [117]:
similar_exist_data = (df[df['STDs: Time since first diagnosis'].notnull()]['STDs: Time since first diagnosis'] == df[df['STDs: Time since last diagnosis'].notnull()]['STDs: Time since last diagnosis'])
print(f'The number of existing data in both cols: {similar_exist_data.shape[0]}')
print(f'The number of similar existing data in both cols: {similar_exist_data.sum()}')

The number of existing data in both cols: 71
The number of similar existing data in both cols: 68


In [118]:
not_equal = similar_exist_data[similar_exist_data == False]
print(list(not_equal.index))
df['STDs: Number of diagnosis'].loc[list(not_equal.index)]

[407, 597, 648]


407    3
597    2
648    2
Name: STDs: Number of diagnosis, dtype: int64

#### 3.5 Minimum value of the time columns

In [119]:
df[['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis']].min()

STDs: Time since first diagnosis    1.0
STDs: Time since last diagnosis     1.0
dtype: float64

#### 3.6 Conclusion and decision

I approved that number of missing values in both columns 'STDs: Time since first diagnosis' and 'STDs: Time since last diagnosis' is equeal (787). It means that the absence of time demonstrates that a person didn't attend a hospital and didn't get a diagnosis.

In 71 existing rows, values aren't equal only in 3 of them (there're several diagnosis that people have). Thus, columns almost copy each other, and the existing of a diagnos showed by 'STDs: Number of diagnosis' column.

The solution: deleating both columns. The lost data is the information about those 3 people (with several diagnosis).

### 4. Side checks (inconclusive)
Early attempts to find a pattern by grouping the gaps by `Number of sexual partners` and comparing with `IUD`. Kept for reference; they did not lead to a decision.

In [120]:
print(df['STDs: Time since first diagnosis'].isnull().groupby(df['Number of sexual partners']).mean() * 100)
print('-'*40)

print(df['Smokes'].isnull().groupby(df['Number of sexual partners']).mean() * 100)

Number of sexual partners
1.0      94.174757
2.0      91.544118
3.0      89.903846
4.0      93.589744
5.0      88.636364
6.0      88.888889
7.0      71.428571
8.0     100.000000
9.0     100.000000
10.0    100.000000
15.0      0.000000
28.0    100.000000
Name: STDs: Time since first diagnosis, dtype: float64
----------------------------------------
Number of sexual partners
1.0       1.456311
2.0       1.470588
3.0       2.403846
4.0       0.000000
5.0       0.000000
6.0       0.000000
7.0       0.000000
8.0       0.000000
9.0       0.000000
10.0      0.000000
15.0    100.000000
28.0      0.000000
Name: Smokes, dtype: float64


In [121]:
(df['STDs: Time since first diagnosis'].isnull() == df['IUD'].isnull()).sum()

np.int64(176)

### 5. Cleaning on a copy

In [122]:
df1 = df.copy()

In [123]:
4# Dropping the rows with low NaN level
missing_columns = [col for col, value in zip(dict(missing_values), missing_values)
                   if 0 < value < 10]


df1[missing_columns] = df1[missing_columns].dropna(axis=0)
print(df1.info())

<class 'pandas.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Age                                 858 non-null    int64  
 1   Number of sexual partners           761 non-null    float64
 2   First sexual intercourse            761 non-null    float64
 3   Num of pregnancies                  761 non-null    float64
 4   Smokes                              761 non-null    float64
 5   Smokes (years)                      761 non-null    float64
 6   Smokes (packs/year)                 761 non-null    float64
 7   Hormonal Contraceptives             750 non-null    float64
 8   Hormonal Contraceptives (years)     750 non-null    float64
 9   IUD                                 741 non-null    float64
 10  IUD (years)                         741 non-null    float64
 11  STDs                                753 non-null    floa

> **TODO:** this cell currently returns an empty DataFrame (`dropna(axis=1)` drops the columns it selected). Rework it so that all columns are kept.

In [124]:
missing_values.sort_values(ascending=False)

STDs: Time since first diagnosis      91.724942
STDs: Time since last diagnosis       91.724942
IUD                                   13.636364
IUD (years)                           13.636364
Hormonal Contraceptives               12.587413
Hormonal Contraceptives (years)       12.587413
STDs:HPV                              12.237762
STDs:AIDS                             12.237762
STDs:Hepatitis B                      12.237762
STDs:HIV                              12.237762
STDs                                  12.237762
STDs:cervical condylomatosis          12.237762
STDs:vulvo-perineal condylomatosis    12.237762
STDs:syphilis                         12.237762
STDs:pelvic inflammatory disease      12.237762
STDs:vaginal condylomatosis           12.237762
STDs:genital herpes                   12.237762
STDs:molluscum contagiosum            12.237762
STDs:condylomatosis                   12.237762
STDs (number)                         12.237762
Num of pregnancies                     6

> **TODO:** `missing_values` was computed on `df`, so this is the old table. Recount the gaps on the cleaned `df1` after the fixes.

### 6. Block of 105 gaps in the STD columns *(in progress)*
Questions: are these the same rows in all 14 columns? What is `Number of diagnosis` in those rows?

In [125]:
std_columns = [col for col, value in zip(dict(missing_values), missing_values)
               if round(value) == 12]
std_columns

['STDs',
 'STDs (number)',
 'STDs:condylomatosis',
 'STDs:cervical condylomatosis',
 'STDs:vaginal condylomatosis',
 'STDs:vulvo-perineal condylomatosis',
 'STDs:syphilis',
 'STDs:pelvic inflammatory disease',
 'STDs:genital herpes',
 'STDs:molluscum contagiosum',
 'STDs:AIDS',
 'STDs:HIV',
 'STDs:Hepatitis B',
 'STDs:HPV']

In [126]:
df1[std_columns].isnull().sum(axis=1).value_counts()

0     753
14    105
Name: count, dtype: int64

> 105 people have all 14 missing STD-columns, the rest - nothing.

In [132]:
df['STDs: Number of diagnosis'] == df1[df1[std_columns].isnull().all(axis=1)][std_columns]

ValueError: Operands are not aligned. Do `left, right = left.align(right, axis=1)` before operating.

In [133]:
df1.shape

(858, 36)

### 7. Monday summary
*(decisions per block of columns and why — to be written)*

---
## Tuesday — Duplicates, wrong types, row-level problems
*(duplicates, type conversion, suspicious values, tidy column names)*

---
## Wednesday — Outliers
*(boxplots, IQR rule, z-score, decisions)*

---
## Thursday — Feature scaling
*(StandardScaler vs MinMaxScaler, fit on train only)*

---
## Friday — Plotly charts
*(bar, histogram, scatter, line)*

---
## Saturday — Mini-project
*(clean dataset + 4 charts + conclusions)*

---
### Scratch
Empty cell with a leftover output (kept, not deleted).